In [10]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
from langchain_community.document_loaders import PyPDFLoader
PDF_PATH="telecom_guide.pdf"
loader=PyPDFLoader(PDF_PATH)
page=loader.load()
print(f"Loaded pdf {len(page)} ")
print("---------first charcter----------")
print(page[0].page_content)

Loaded pdf 9 
---------first charcter----------
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

spliter = RecursiveCharacterTextSplitter(
            chunk_size=600,
            chunk_overlap=100,
            separators=["\n\n","\n","."," "],
)
chunk=spliter.split_documents(page)
len(chunk)

37

In [13]:
chunk[0].page_content

'Telecom Technical Reference Guide  - Internal Use Only\nTelecom Technical\nReference Guide\nCustomer Care & Network Operations Edition\nVersion 3.2  |  Covers 2G / 3G / 4G LTE / 5G\nPage 1'

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embading=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store=Chroma.from_documents(chunk,embading)
print(f"Vector store created with {vector_store._collection.count()} vectors")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 454.82it/s]


Vector store created with 74 vectors


In [15]:
retrierver=vector_store.as_retriever(search_kwargs={"k":3})
test_query="What is the VoLTE and how does it work?"
retrived=retrierver.invoke(test_query)
for i,doc in enumerate(retrived):
    print(f"Document {i+1}:")
    print(doc.page_content[:300])
    print("--------------------")

Document 1:
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt
--------------------
Document 2:
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Opt
--------------------
Document 3:
Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls 
-------------------

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

SYSTEM_PROMPT="""
You are a telecom expert. You have been asked to answer questions related to telecom technology. 
Use the provided context to answer the question. If the answer is not contained within the context,
 say "I don't know". Do not try to make up an answer.
 Context:
 {context}
"""
prompt=ChatPromptTemplate.from_messages([
    ("system",SYSTEM_PROMPT),
    ("human","{question}")
])

llm = ChatGroq(model="groq/compound-mini", temperature=0)


chain=(
    {"context":retrierver | format_docs,"question":RunnablePassthrough()}
    | prompt
    | llm
)
print("RAG chain executed successfully.")

RAG chain executed successfully.


In [17]:
question="What is the VoLTE and how does it work?"
print(f"Question: {question}")
print("Answer:")
print(chain.invoke(question))

Question: What is the VoLTE and how does it work?
Answer:
content='**VoLTE (Voice over LTE)** is an IP‑based voice service that lets you make phone calls over the LTE data network instead of using the older circuit‑switched voice channels found in 2G and 3G networks.\n\n**How it works**\n\n1. **IMS core** – Calls are carried as data packets through the LTE network using the IP Multimedia Subsystem (IMS) core.  \n2. **Packet‑switched transmission** – Voice is treated like any other IP traffic, so the same LTE radio that carries your internet data also carries the voice packets.  \n3. **Benefits**  \n   * **HD voice quality** – Wide‑band audio (≈16\u202fkHz) compared with the ~3.4\u202fkHz of legacy calls.  \n   * **Faster call setup** – Calls connect in under 2\u202fseconds, versus 6–8\u202fseconds on 3G.  \n   * **Simultaneous voice\u202f+\u202fdata** – You can talk and use data (e.g., browse the web) at the same time without degradation.\n\n**Requirements**\n\n* A VoLTE‑compatible dev